# Laboratorium 10

**Imię i nazwisko:** Daniel Stefański  
**Nazwa ćwiczenia:** GAN i generowanie cyfr MNIST

### Zadanie.1

Korzystając ze szkieletów klas z załączonego pliku, zaimplementuj sieci Generator i Discriminator: 

    • Użyj architektury MLP: warstwy nn.Linear, aktywacje nn.LeakyReLU(0.2), 
    nn.Dropout(0.3) w Dyskryminatorze. Na wyjściu Generatora nn.Tanh(), na wyjściu 
    Dyskryminatora nn.Sigmoid(). 
    • Przetestuj kształty tensorów: wygeneruj z = torch.randn(16, 100) i sprawdź, że G(z) 
    ma kształt (16, 784) oraz D(G(z)) ma kształt (16, 1). 
    • Oblicz liczbę parametrów obu sieci (sum(p.numel() for p in 
    model.parameters())).  
    • W komórce Markdown odpowiedz: ile parametrów ma Generator, ile Dyskryminator? 
    Która warstwa ma ich najwięcej?

In [1]:
import torch
import torch.nn as nn

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, img_dim),
            nn.Tanh(),
        )
 
    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, img_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )
 
    def forward(self, img):
        return self.net(img)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

latent_dim = 100
G = Generator(latent_dim)
D = Discriminator()

z = torch.randn(16, 100)
fake_imgs = G(z)
disc_output = D(fake_imgs)

print(f"Ksztalt G(z): {tuple(fake_imgs.shape)}")
print(f"Ksztalt D(G(z)): {tuple(disc_output.shape)}")
print(f"Liczba parametrow Generatora: {count_params(G)}")
print(f"Liczba parametrow Dyskryminatora: {count_params(D)}")


Generator ma `1486352` parametr?w, a Dyskryminator `533505`. W obu sieciach najwi?cej parametr?w znajduje si? w najwi?kszych warstwach liniowych. W Generatorze najbardziej kosztowna jest ostatnia warstwa `Linear(1024, 784)`, a w Dyskryminatorze pierwsza warstwa `Linear(784, 512)`. To jest naturalne, bo liczba parametr?w w warstwie liniowej ro?nie wraz z iloczynem liczby wej?? i wyj??.


### Zadanie.2

Uzupełnij pętlę treningu ze szkieletu w sekcji 2 i wytrenuj GAN przez 50 epok: 

    • Krok Dyskryminatora: oblicz stratę na próbkach prawdziwych (torch.ones) i fałszywych 
    (torch.zeros, pamiętaj o .detach()). Wykonaj krok opt_D. 
    • Krok Generatora: wygeneruj nowe próbki, oblicz stratę z etykietą torch.ones. Wykonaj 
    krok opt_G. 
    • Co 10 epok wyświetl siatkę 64 wygenerowanych obrazów 
    (torchvision.utils.make_grid). Po treningu narysuj wykres wartości L_D i L_G na 
    jednym wykresie. 
    • W komórce Markdown opisz: kiedy wygenerowane cyfry stają się po raz pierwszy 
    rozpoznawalne? Jak zachowują się straty w kolejnych epokach?

In [ ]:
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

if hasattr(torchvision.datasets.MNIST, 'mirrors'):
    torchvision.datasets.MNIST.mirrors = ['https://ossci-datasets.s3.amazonaws.com/mnist/']

if hasattr(torchvision.datasets.MNIST, 'urls'):
    torchvision.datasets.MNIST.urls = [
        'https://ossci-datasets.s3.amazonaws.com/mnist/train-images-idx3-ubyte.gz',
        'https://ossci-datasets.s3.amazonaws.com/mnist/train-labels-idx1-ubyte.gz',
        'https://ossci-datasets.s3.amazonaws.com/mnist/t10k-images-idx3-ubyte.gz',
        'https://ossci-datasets.s3.amazonaws.com/mnist/t10k-labels-idx1-ubyte.gz'
    ]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Urzadzenie: {device}')

latent_dim = 100
lr = 2e-4
G = Generator(latent_dim).to(device)
D = Discriminator().to(device)
criterion = nn.BCELoss()
opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

losses_G = []
losses_D = []
fixed_z = torch.randn(64, latent_dim, device=device)

for epoch in range(50):
    epoch_ld = 0.0
    epoch_lg = 0.0
    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.view(-1, 784).to(device)
        bs = real_imgs.size(0)

        real_labels = torch.ones(bs, 1, device=device)
        fake_labels = torch.zeros(bs, 1, device=device)

        # Krok 1 - trenuj Dyskryminator
        z = torch.randn(bs, latent_dim, device=device)
        fake_imgs = G(z)
        opt_D.zero_grad()
        loss_real = criterion(D(real_imgs), real_labels)
        loss_fake = criterion(D(fake_imgs.detach()), fake_labels)
        loss_D = loss_real + loss_fake
        loss_D.backward()
        opt_D.step()

        # Krok 2 - trenuj Generator
        z = torch.randn(bs, latent_dim, device=device)
        generated = G(z)
        opt_G.zero_grad()
        loss_G = criterion(D(generated), real_labels)
        loss_G.backward()
        opt_G.step()

        epoch_ld += loss_D.item()
        epoch_lg += loss_G.item()

    epoch_ld /= len(dataloader)
    epoch_lg /= len(dataloader)
    losses_D.append(epoch_ld)
    losses_G.append(epoch_lg)
    print(f'Epoka {epoch+1}: L_D={epoch_ld:.4f}, L_G={epoch_lg:.4f}')

    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            samples = G(fixed_z).view(-1, 1, 28, 28)
            samples = (samples + 1) / 2
            grid = make_grid(samples, nrow=8)
            plt.figure(figsize=(6, 6))
            plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
            plt.axis('off')
            plt.title(f'Probki po epoce {epoch+1}')
            plt.show()

plt.figure(figsize=(8, 5))
plt.plot(range(1, 51), losses_D, label='L_D')
plt.plot(range(1, 51), losses_G, label='L_G')
plt.xlabel('Epoka')
plt.ylabel('Strata')
plt.title('Straty GAN podczas treningu')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


W tym eksperymencie pierwsze cyfry zaczynaj? by? rozpoznawalne ju? oko?o **10 epoki**, ale s? wtedy jeszcze mocno rozmyte i niestabilne. Oko?o **20 epoki** wi?kszo?? wygenerowanych obraz?w ma ju? wyra?ny kszta?t cyfr, a dalszy trening poprawia g??wnie czytelno?? i sp?jno?? pr?bek.

Straty zachowuj? si? typowo dla GAN-a: `L_G` na pocz?tku jest wysoka, potem stopniowo maleje i pod koniec treningu stabilizuje si? w okolicach **0.86**. Z kolei `L_D` po pocz?tkowych wahaniach ro?nie i stabilizuje si? w okolicach **1.29**. Nie oznacza to b??du ? w GAN-ach wa?niejsza od monotonicznego spadku strat jest r?wnowaga mi?dzy generatorem i dyskryminatorem oraz jako?? wygenerowanych pr?bek.

Zapisane wyniki z uruchomienia:
- `L_D` po 10 epoce: `1.1981`, `L_G`: `1.0112`
- `L_D` po 20 epoce: `1.2598`, `L_G`: `0.9214`
- `L_D` po 30 epoce: `1.2783`, `L_G`: `0.8849`
- `L_D` po 40 epoce: `1.2829`, `L_G`: `0.8731`
- `L_D` po 50 epoce: `1.2900`, `L_G`: `0.8601`

Pr?bki i wykresy z treningu:

![Epoka 10](screen/gan_epoch_10.png)

![Epoka 20](screen/gan_epoch_20.png)

![Epoka 30](screen/gan_epoch_30.png)

![Epoka 40](screen/gan_epoch_40.png)

![Epoka 50](screen/gan_epoch_50.png)

![Straty GAN](screen/gan_losses.png)


### Zadanie.3

Eksploracja przestrzeni latentnej: 

    • Wygeneruj 100 różnych próbek (z = torch.randn(100, 100)) i wyświetl je jako 
    siatkę 10×10. 
    • Wykonaj interpolację liniową między dwoma wylosowanymi punktami z_1 i z_2: 
    wygeneruj 10 punktów pośrednich i wyświetl odpowiadające im obrazy w jednym rzędzie 
    (hint: z = (1-t)*z1 + t*z2 dla t = 0, 0.1, ..., 1.0). 
    • W komórce Markdown: co obserwujesz przy interpolacji? Czy przejścia między cyframi są 
    gładkie? Co to mówi o strukturze przestrzeni latentnej?

In [ ]:
with torch.no_grad():
    z = torch.randn(100, latent_dim, device=device)
    samples = G(z).view(-1, 1, 28, 28)
    samples = (samples + 1) / 2
    grid_100 = make_grid(samples, nrow=10)

plt.figure(figsize=(8, 8))
plt.imshow(grid_100.permute(1, 2, 0).cpu().numpy(), cmap='gray')
plt.axis('off')
plt.title('100 wygenerowanych probek (10x10)')
plt.show()

with torch.no_grad():
    z1 = torch.randn(1, latent_dim, device=device)
    z2 = torch.randn(1, latent_dim, device=device)
    interpolations = []
    for i in range(10):
        t = i / 9
        z_interp = (1 - t) * z1 + t * z2
        img = G(z_interp).view(1, 1, 28, 28)
        interpolations.append(img)
    interpolations = torch.cat(interpolations, dim=0)
    interpolations = (interpolations + 1) / 2
    grid_interp = make_grid(interpolations, nrow=10)

plt.figure(figsize=(12, 2.5))
plt.imshow(grid_interp.permute(1, 2, 0).cpu().numpy(), cmap='gray')
plt.axis('off')
plt.title('Interpolacja liniowa w przestrzeni latentnej')
plt.show()


W siatce 100 pr?bek wida?, ?e generator nauczy? si? produkowa? r??ne cyfry o do?? wyra?nych kszta?tach, chocia? cz??? z nich nadal jest lekko rozmyta albo ma nietypowy styl pisma. To sugeruje, ?e model nauczy? si? podstawowej struktury zbioru MNIST, ale nie jest jeszcze idealny.

Przy interpolacji przej?cia mi?dzy kolejnymi obrazami s? do?? g?adkie. Nie ma nag?ego przeskoku z jednego obrazu do zupe?nie losowego szumu, tylko stopniowa zmiana kszta?tu cyfry i jej stylu. Na zapisanym przyk?adzie wida? p?ynne przej?cie mi?dzy podobnymi kszta?tami, co sugeruje, ?e przestrze? latentna jest uporz?dkowana i ci?g?a.

To m?wi nam, ?e generator nie zapami?tuje tylko pojedynczych przyk?ad?w treningowych, ale uczy si? bardziej og?lnej reprezentacji cyfr. Punkty le??ce blisko siebie w przestrzeni latentnej odpowiadaj? podobnym obrazom, co jest jedn? z wa?nych w?asno?ci dobrze wytrenowanego GAN-a.

Wygenerowane obrazy do zadania 3:

![100 probek](screen/gan_100_samples.png)

![Interpolacja](screen/gan_interpolation.png)
